# [검증] 집계구 유동인구-기온 계절성 — 2026-10-04

- 작성자: 이정연 (`ljy`)
- 목적: 12월 SK 유동인구 감소가 기온(한파) 때문인지, 기온과 무관하게 지속되는 악화인지 1차 진단
- 입력: `data/raw/sk_통신인구_202507~202512_통합.csv`(6개월), `data/raw/reference_data/기상/`의 Open-Meteo ERA5 일별 기온
- 대상 시군구: 강남구(11230, 집계구 기준 "서울")·춘천시(32010, 집계구 기준 "춘천")
- **선행 분석**: `notebooks/260923_EDA_external_factors.ipynb`가 카드·기상·전철을 지역 단위로 결합했다. 이 노트북은 그 다음 단계를 SK 유동인구 + 집계구 단위로 수행한다.
- **한계**: 상관계수 하나로 "순수 계절효과다/아니다"를 확정하지 않는다. 기온도 7~12월 사이 단조 감소하므로, 유동인구가 기온과 무관한 이유로 단조 감소해도 비슷한 상관이 나올 수 있다. 같은 집계구의 6개 관측치는 독립이 아니라서 p-value는 참고용이다. ERA5 서울·춘천 지점은 구·시 전체의 대리변수이지 집계구별 관측이 아니다.

In [1]:
from pathlib import Path

import pandas as pd

from src.data.weather import load_monthly_temperature
from src.features.block_panel import aggregate_to_block, load_oa_boundaries
from src.features.flow_temperature import (
    correlate_flow_with_temperature,
    flow_deviation,
    monthly_log_flow,
    temperature_deviation,
)

MONTHS = ["202507", "202508", "202509", "202510", "202511", "202512"]

repo_root = Path.cwd()
if not (repo_root / "data" / "raw").exists():
    repo_root = repo_root.parent
raw_dir = repo_root / "data" / "raw"

boundaries = load_oa_boundaries(raw_dir / "reference_data")
frames = [
    pd.read_csv(
        raw_dir / f"sk_통신인구_{month}_통합.csv",
        dtype={"STD_YM": "string", "BLOCK_CD": "string"},
    )
    for month in MONTHS
]
combined = pd.concat(frames, ignore_index=True)
panel = aggregate_to_block(combined, boundaries)
print(f"집계구×월 패널 행: {len(panel):,}, 고유 집계구 수: {panel['집계구'].nunique():,}")

집계구×월 패널 행: 10,170, 고유 집계구 수: 1,695


In [2]:
flow = monthly_log_flow(panel)
flow_dev = flow_deviation(flow)

temp = load_monthly_temperature(raw_dir / "reference_data" / "기상")
temp_dev = temperature_deviation(temp)
print("지역별 월평균기온(편차):")
print(temp_dev.round(2).to_string(index=False))

result = correlate_flow_with_temperature(flow_dev, temp_dev)
print("\n지역별 유동인구-기온 편차 Spearman 상관:")
print(result.round(4).to_string())

지역별 월평균기온(편차):
지역 STD_YM  평균기온  평균기온_편차
서울 202507 27.74    10.70
서울 202508 27.33    10.29
서울 202509 22.82     5.78
서울 202510 16.18    -0.86
서울 202511  7.86    -9.18
서울 202512  0.31   -16.73
춘천 202507 27.85    10.55
춘천 202508 27.74    10.43
춘천 202509 22.84     5.54
춘천 202510 16.20    -1.10
춘천 202511  8.14    -9.17
춘천 202512  1.04   -16.26

지역별 유동인구-기온 편차 Spearman 상관:
    correlation  p_value     n
지역                            
서울       0.2719      0.0  6576
춘천       0.1756      0.0  3594


## 결론 (2026-10-04 fresh reviewer 피드백 반영해 수정)

- **서울(강남) r=0.272, 춘천 r=0.176 — 둘 다 양(+)의 상관이고 통계적으로 유의하게 나온다.** 다만 이 분석이 실제로 보여주는 건 처음 의도했던 것보다 좁다. 아래 두 가지를 바로잡는다.

- **이 설계로는 "기온의 효과"와 "그냥 달이 지나가는 효과(추세)"를 구분할 수 없다.** 기온이 서울·춘천 둘 다 7월→12월로 엄격하게 단조 감소하기 때문에(27.74→0.31, 27.85→1.04), 같은 6개 지역-월에서 "유동인구 편차 vs 기온 편차"의 Spearman 상관은 "유동인구 편차 vs 월 순서(1~6)"의 Spearman 상관과 수학적으로 똑같다. 그러니까 r=0.272라는 숫자가 확인해주는 건 "유동인구가 기온을 따라간다"가 아니라 "유동인구가 7~12월 동안 대체로 단조롭게 움직인다(이 기간엔 주로 감소)"는 것뿐이다. **"기온 영향이 섞여 있다는 걸 확인했다"는 이전 버전의 문장은 틀렸다 — 삭제한다.**
- **r이 1에 가깝지 않다고 해서 "기온으로 설명되지 않는 부분이 많다"는 뜻도 아니다.** 집계구 1,695개를 풀링한 Spearman은 집계구마다 편차의 크기(scale)가 서로 다르고, 기온 쪽 값은 지역당 6개 수준으로만 묶여 있어서(한 달에 수백 개 집계구가 똑같은 기온값을 공유) 애초에 1에 가까워지기 어려운 구조다. 두 집계구가 기온과 완벽하게 단조로 움직여도 크기만 다르면 풀링한 r은 1보다 한참 낮게 나올 수 있다. **"상당 부분이 설명 안 된다"는 이전 버전의 해석도 틀렸다 — 삭제한다.**
- **반복측정이라 p-value를 "강한 증거"로 읽으면 안 된다.** 같은 집계구의 6개 관측치는 서로 독립이 아닌데 n=6,576/3,594로 풀링해서 검정했다 — 유의성이 과장돼 보인다. "양의 상관이 유의하게 있다"는 표현도 쓰지 않는다. 상관계수의 방향과 대략적 크기만 참고한다.
- **ERA5 기온은 집계구별 관측이 아니라 서울·춘천 각각 하나의 대리 지점이다** — 강남 안의 모든 집계구에 똑같은 서울 기온값을 적용했다. 집계구 간 실제 기온 차이는 반영되지 않는다.
- **범위도 다시 분명히 한다**: 원래 질문은 "12월 감소가 한파 때문인가"였는데, 이 분석은 12월만 따로 떼어보지 않고 7~12월 전체를 풀링한 상관이다. 12월에 국한된 질문에는 직접 답하지 못한다.

- **그래서 실제로 확인된 것**: 유동인구가 7~12월 동안 대체로 단조롭게 감소하는 패턴이 있다는 것. 이게 "한파 때문"인지 "기온과 무관한 진짜 악화"인지는 **이 6개월·1개 연도 자료로는 원리상 구분이 안 된다** — 기온과 "달이 지나가는 효과"가 완전히 같이 움직이기 때문이다(공선성이 아니라 식별 불가능).
- **"12월 스냅샷이냐 6개월 평균이냐" 질문에는 이 분석이 직접적인 근거를 주지 못한다.** 원래 "평균 쪽에 힘을 실어준다"고 썼던 잠정 결론도 철회한다 — 기온 효과를 확인하지 못했으니 그 근거가 없다.
- **다음에 시도할 만한 방향** (월 순서와의 상관 비교는 위에서 설명했듯 같은 숫자만 나오므로 제외):
  - 월내 일별·주별 기온 변동과 유동인구 변동을 보면(월 단위보다 더 잘게 쪼개면) 기온과 달력 효과가 어느 정도 분리된다.
  - 서울-춘천 간 기온 차이(두 지역이 완전히 같은 패턴으로 움직이지 않는 부분)를 활용하면 지역 공통의 달력 효과를 어느 정도 통제할 수 있다.
  - 전년도 동월 데이터가 있으면 "같은 달, 다른 해"로 기온과 달력을 분리할 수 있다 — 지금은 2025년 한 해치만 있어서 불가능하다.
  - 강수·적설은 아직 안 봤다.